# 01 - Préparation des données (CMS Open Payments)

Téléchargement via l'API CMS (un État, deux années), nettoyage, audit qualité,
agrégation au niveau du professionnel de santé et construction de la cible de
rétention. Sortie : `data/openpayments.sqlite` (tables `payments` et `hcp_features`).

In [ ]:
import sqlite3, pathlib
import pandas as pd
import requests

API = "https://openpaymentsdata.cms.gov/api/1"
STATE = "WV"
YEAR_FEATURES, YEAR_TARGET = 2022, 2023
DATASET_IDS = {2022: "df01c2f8-dc1f-4e79-96cb-8208beaf143c",
               2023: "fb3a65aa-c901-4a38-a813-b04b00dfa2a9",
               2024: "e6b17c6a-2534-4207-a4a1-6746a14911ff"}
KEEP = ["covered_recipient_profile_id", "covered_recipient_type", "recipient_state",
        "covered_recipient_specialty_1",
        "applicable_manufacturer_or_applicable_gpo_making_payment_name",
        "total_amount_of_payment_usdollars", "nature_of_payment_or_transfer_of_value", "program_year"]
PAGE = 500  # l'API CMS plafonne limit a 500

ROOT = pathlib.Path.cwd().parent if (pathlib.Path.cwd().parent / "data").exists() else pathlib.Path.cwd()
DB_PATH = ROOT / "data" / "openpayments.sqlite"
print("Projet:", ROOT)

## Récupération via l'API CMS (pagination par 500)

In [ ]:
def fetch_state_year(year, state):
    url = f"{API}/datastore/query/{DATASET_IDS[year]}/0"
    rows, offset, total = [], 0, None
    while True:
        params = {"limit": PAGE, "offset": offset,
                  "conditions[0][property]": "recipient_state",
                  "conditions[0][value]": state,
                  "conditions[0][operator]": "="}
        payload = requests.get(url, params=params, timeout=120).json()
        batch = payload.get("results", [])
        if total is None:
            total = payload.get("count", 0); print(f"  {year}/{state}: {total} lignes")
        if not batch:
            break
        rows.extend(batch); offset += len(batch)
        if offset >= total:
            break
    df = pd.DataFrame(rows)
    return df[[c for c in KEEP if c in df.columns]].copy()

## Nettoyage et audit qualité

In [ ]:
def clean(df):
    df = df.copy()
    df["total_amount_of_payment_usdollars"] = pd.to_numeric(df["total_amount_of_payment_usdollars"], errors="coerce")
    df["program_year"] = pd.to_numeric(df["program_year"], errors="coerce").astype("Int64")
    df["specialty"] = df["covered_recipient_specialty_1"].fillna("Unknown").str.split("|").str[0].str.strip()
    df = df[df["covered_recipient_profile_id"].notna()]
    df = df[df["covered_recipient_profile_id"].astype(str).str.len() > 0]
    return df

def quality_audit(df, label):
    print(f"=== Audit qualite - {label} ===")
    print("  lignes                :", len(df))
    print("  professionnels uniques:", df["covered_recipient_profile_id"].nunique())
    print("  montant manquant      :", int(df["total_amount_of_payment_usdollars"].isna().sum()))
    print("  montant total (USD)   :", round(df["total_amount_of_payment_usdollars"].sum()))

## Agrégation : profil d'engagement + cible de rétention

In [ ]:
def build_features(df_feat, ids_target):
    g = df_feat.groupby("covered_recipient_profile_id")
    feats = pd.DataFrame({
        "n_payments": g.size(),
        "total_amount": g["total_amount_of_payment_usdollars"].sum(),
        "mean_amount": g["total_amount_of_payment_usdollars"].mean(),
        "n_manufacturers": g["applicable_manufacturer_or_applicable_gpo_making_payment_name"].nunique(),
        "n_natures": g["nature_of_payment_or_transfer_of_value"].nunique(),
        "specialty": g["specialty"].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else "Unknown"),
        "state": g["recipient_state"].first(),
    })
    for nature, col in [("Food and Beverage", "share_food"), ("Travel and Lodging", "share_travel"),
                        ("Consulting Fee", "share_consulting"),
                        ("Compensation for services other than consulting, including serving as faculty or as a speaker at a venue other than a continuing education program", "share_speaker"),
                        ("Education", "share_education")]:
        part = df_feat[df_feat["nature_of_payment_or_transfer_of_value"] == nature].groupby("covered_recipient_profile_id").size()
        feats[col] = (part / feats["n_payments"]).reindex(feats.index).fillna(0.0)
    feats["retenu"] = feats.index.to_series().isin(ids_target).astype(int)
    return feats.reset_index()

## Exécution (télécharge, agrège, écrit la base SQLite)

In [ ]:
df_n = clean(fetch_state_year(YEAR_FEATURES, STATE))
df_n1 = clean(fetch_state_year(YEAR_TARGET, STATE))
quality_audit(df_n, f"{STATE} {YEAR_FEATURES}")
quality_audit(df_n1, f"{STATE} {YEAR_TARGET}")

ids_target = set(df_n1["covered_recipient_profile_id"].unique())
features = build_features(df_n, ids_target)
print("Taux de retention:", round(features["retenu"].mean(), 3))

payments = pd.concat([df_n, df_n1], ignore_index=True)
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
with sqlite3.connect(DB_PATH) as con:
    payments.to_sql("payments", con, if_exists="replace", index=False)
    features.to_sql("hcp_features", con, if_exists="replace", index=False)
print("Base ecrite:", DB_PATH, "|", len(payments), "paiements,", len(features), "profils")